# 🎯 Career Fit Assistant: CV-to-Job Match Analyzer

A personal AI career assistant that evaluates how well your CV matches a specific job description, giving you an honest **fit score out of 10** along with actionable feedback.

## What it does

- Takes two inputs: your **CV** (as text or PDF) and a **job description** (pasted text or URL)
- Uses a carefully crafted **system prompt** to instruct the LLM to act as an experienced technical recruiter/career coach
- Analyzes alignment across skills, experience, and keywords between the CV and job requirements
- Returns a **numerical fit rating (0–10)** with a clear breakdown of strengths, gaps, and specific suggestions to improve alignment

## Why this matters

Instead of manually guessing whether you're a strong candidate for a role, this tool gives instant, structured feedback — useful for tailoring applications before you apply, similar to how an ATS resume optimizer works but with qualitative reasoning instead of just keyword matching.


## How to use

1. Paste your CV text into the `cv` variable
2. Paste the target job description into the `job_description` variable
3. Run the notebook — the model returns a fit score and detailed reasoning

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file!")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
MODEL = "gemini-3.5-flash-lite"

API key found and looks good so far!


In [2]:
cv = """
Jordan Lee
Software Engineer

Experience:
- 3 years as a Backend Developer at a fintech startup, building REST APIs in Python (Flask, FastAPI)
- Worked with PostgreSQL, Docker, and basic AWS (EC2, S3)
- Led a small team migrating a monolith to microservices

Skills: Python, SQL, Docker, Git, REST APIs, unit testing

Education: B.Sc. in Computer Science
"""

job_description = """
We are hiring a Senior Backend Engineer to join our platform team.

Requirements:
- 5+ years of experience building scalable backend systems
- Strong experience with Python and cloud infrastructure (AWS or GCP)
- Experience with Kubernetes and CI/CD pipelines
- Familiarity with event-driven architecture (Kafka or similar)
- Experience mentoring junior engineers

Nice to have: experience with Go, Terraform
"""

In [3]:
system_prompt = """You are an experienced technical recruiter and career coach.
You will be given a candidate's CV and a target job description.
Compare them carefully across skills, experience, and keywords, and assess how well the candidate fits the role.

Respond with ONLY a valid JSON object, with no extra commentary, explanation, or markdown formatting.
The JSON object must have exactly these fields:
- "score": an integer from 0 to 10 rating the overall fit
- "strengths": a list of short strings describing what matches well
- "gaps": a list of short strings describing what is missing or weak
- "suggestions": a list of short strings with concrete suggestions to improve the fit

Return ONLY the JSON object, nothing else."""

In [4]:
def build_user_prompt(cv, job_description):
    return f"""Here is the candidate's CV:

{cv}

Here is the job description:

{job_description}

Please assess the fit and respond with the JSON object as instructed."""

In [5]:
def get_fit_score(cv, job_description):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_user_prompt(cv, job_description)}
    ]

    response = gemini.chat.completions.create(
        model=MODEL,
        messages=messages,
    )

    raw_content = response.choices[0].message.content

    try:
        return json.loads(raw_content)
    except json.JSONDecodeError:
        print("Failed to parse JSON. Raw response was:")
        print(raw_content)
        raise

In [6]:
result = get_fit_score(cv, job_description)

print(f"Fit score: {result['score']}/10\n")

print("Strengths:")
for item in result["strengths"]:
    print(f"- {item}")

print("\nGaps:")
for item in result["gaps"]:
    print(f"- {item}")

print("\nSuggestions:")
for item in result["suggestions"]:
    print(f"- {item}")

Fit score: 4/10

Strengths:
- Relevant background in Python and backend development
- Experience with microservices migration, Docker, and AWS
- Holds a B.Sc. in Computer Science

Gaps:
- Only 3 years of experience compared to the required 5+
- Missing experience with Kubernetes and CI/CD pipelines
- No experience with event-driven architecture (Kafka)
- Lacks advanced cloud infrastructure (Terraform) and Go
- Experience is focused on small team leadership rather than senior-level engineering and mentoring

Suggestions:
- Highlight any hands-on exposure to CI/CD pipelines or container orchestration if applicable
- Gain familiarity with event-driven messaging systems like Kafka or RabbitMQ
- Learn Kubernetes and Infrastructure as Code tools like Terraform
- Emphasize mentorship of junior developers during past team leadership
